In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 7 — Ejercicio 1
# ---------------------------------------------------------------

# Adaptando el código del Paso 5 para graficar la curva de sobreajuste
profundidades = range(1, 16) # Profundidades del 1 al 15
acc_train = []
acc_test = []

# DecisionTreeClassifier se importa en el Capítulo 7, junto con accuracy_score y matplotlib.pyplot
for d in profundidades:
    dt = DecisionTreeClassifier(max_depth=d, random_state=semilla)
    dt.fit(X_train, y_train)
    acc_tr = accuracy_score(y_train, dt.predict(X_train))
    acc_te = accuracy_score(y_test,  dt.predict(X_test))
    acc_train.append(acc_tr)
    acc_test.append(acc_te)
    print(f'depth={d:2d}  train={acc_tr:.4f}  test={acc_te:.4f}')

# Graficar los resultados
plt.figure(figsize=(10, 6))
plt.plot(profundidades, acc_train, label='Accuracy en Train', marker='o')
plt.plot(profundidades, acc_test, label='Accuracy en Test', marker='x')

plt.title('Curva de Sobreajuste para Árbol de Decisión')
plt.xlabel('Profundidad máxima del árbol (max_depth)')
plt.ylabel('Accuracy')
plt.xticks(profundidades)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Encontrar el punto de cruce (aproximado o donde test deja de mejorar)
# Visualmente se puede observar en el gráfico.
# Si se necesita un valor exacto, se puede iterar sobre los resultados.
mejor_acc_test = max(acc_test)
mejor_profundidad = profundidades[acc_test.index(mejor_acc_test)]
print(f"\nEl mejor accuracy en test ({mejor_acc_test:.4f}) se obtiene con max_depth={mejor_profundidad}")


In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 7 — Ejercicio 2
# ---------------------------------------------------------------
# Declaraciones e importaciones ocurren en el capítulo 7

n_estimators_list = [10, 50, 100, 200, 500]
max_features_list = ['sqrt', 'log2', 0.5]

resultados_rf = []
mejor_acc = 0
mejor_params = {}

print(f'{'n_estimators':<12} {'max_features':<15} {'Test Accuracy':<15}')
print(f'{'-'*12:<12} {'-'*15:<15} {'-'*15:<15}')

for n in n_estimators_list:
    for mf in max_features_list:
        # Crear el modelo Random Forest con los hiperparámetros actuales
        rf_model = RandomForestClassifier(
            n_estimators=n,
            max_features=mf,
            random_state=semilla,
            n_jobs=-1
        )

        # Entrenar el modelo
        rf_model.fit(X_train, y_train)

        # Predecir y calcular accuracy en test
        y_pred_test = rf_model.predict(X_test)
        acc = accuracy_score(y_test, y_pred_test)

        # Guardar resultados
        resultados_rf.append({
            'n_estimators': n,
            'max_features': mf,
            'test_accuracy': acc
        })

        print(f'{n:<12} {str(mf):<15} {acc:.4f}')

        # Actualizar el mejor resultado si es necesario
        if acc > mejor_acc:
            mejor_acc = acc
            mejor_params = {'n': n, 'mf': mf}

print(f'\nMejor Accuracy en Test: {mejor_acc:.4f}')
print(f'Mejores hiperparámetros: n_estimators={mejor_params['n']}, max_features={mejor_params['mf']}')


In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 7 — Ejercicio 3
# ---------------------------------------------------------------

from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report
import numpy as np
import matplotlib.pyplot as plt

cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
print("Distribución:", np.bincount(y), "→ [maligno, benigno]")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe = Pipeline([
    ("imp",   SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
])

# Validación cruzada en train
cv_scores = cross_val_score(pipe, X_train, y_train, cv=5)
print(f"CV mean: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Entrenamiento y evaluación final
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

# Feature importances (top 10)
importancias = pipe.named_steps["model"].feature_importances_
top10_idx = np.argsort(importancias)[::-1][:10]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(cancer.feature_names[top10_idx][::-1],
        importancias[top10_idx][::-1])
ax.set_title("Top 10 features — Random Forest Breast Cancer")
plt.tight_layout()
plt.show()
